# variableMeanSpatialNoise

Standalone notebook for the `VariableMeanSpatialNoise` protocol. Use this for real VM-noise experiments — pick a date that ran the protocol and the same pipeline (StimBlock + ResponseBlock + nearest noise chunk via `create_mea_pipeline`) applies.

Below is a synthetic demo path used to visualize what the noise stimulus looks like overlaid on the recording array. It's useful as a sanity check for geometry (canvas → MEA chip) when no real VM-noise data is available; replace the synthetic `stim_block` with a real one once you have a recording.

In [1]:
import os
import retinanalysis as ra
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 1. Build a pipeline

For the synthetic VM-noise demo we just need an `AnalysisChunk` to borrow its rig geometry (`canvas_size`, `microns_per_pixel`) and its mosaic. We piggy-back on whatever experiment is first in the protocol registry. Swap `PROTOCOL_SEARCH` / `exp_name` / `datafile_name` for a real VM-noise recording when ready.

In [7]:
# Protocol-registry rows for this protocol, filtered to dates with
# Kilosort output on disk. ra.find_available_datasets handles both steps
# (DJ query + os.listdir intersect) so §1 and §9 stay in sync.
exp_search = ra.find_available_datasets('monitorVariableMeanNoiseEpochs')

print(f'{len(exp_search)} usable datafile(s) found across '
      f'{exp_search.exp_name.nunique()} experiment(s).')
display(exp_search)



Found 2 protocols matching "monitorvariablemeannoiseepochs":
['edu.washington.riekelab.chris.protocols.monitorVariableMeanNoiseEpochs'
 'edu.washington.riekelab.vyom.protocols.monitorVariableMeanNoiseEpochs']

Found 16 experiments, 17 epoch blocks.

13 usable datafile(s) found across 12 experiment(s).


,exp_name,datafile_name,NDF,chunk_name,protocol_name,is_mea,data_dir,group_label,experiment_id,protocol_id,group_id,block_id,chunk_id
0,20230214C,data020,3.0,chunk3,edu.washington.riekelab.vyom.protocols.monitor...,1,20230214C/data020,Var Mean Noise ndf 3.0,39,40,783,1486,99
1,20230313C,data002,3.0,chunk1,edu.washington.riekelab.vyom.protocols.monitor...,1,20230313C/data002,Var mean noise ndf 3; misaligned light path,42,40,867,1595,107
2,20230502C,data016,3.0,chunk2,edu.washington.riekelab.vyom.protocols.monitor...,1,20230502C/data016,ndf 3.0 var mean noise,52,40,1075,1853,156
3,20231220C,data017,0.5,chunk3,edu.washington.riekelab.vyom.protocols.monitor...,1,20231220C/data017,CC cone var mean noise,76,40,1602,2512,248
4,20240117C,data029,0.5,chunk4,edu.washington.riekelab.vyom.protocols.monitor...,1,20240117C/data029,CC var mean noise,77,40,1636,2550,253
5,20240130C,data015,0.5,chunk4,edu.washington.riekelab.vyom.protocols.monitor...,1,20240130C/data015,CC cone var mean nosie short epoch,78,40,1653,2568,261
6,20240523C,data023,0.5,dynamics2,edu.washington.riekelab.vyom.protocols.monitor...,1,20240523C/data023,CC cone var mean noise,93,40,1838,2806,300
7,20240523C,data012,0.5,dynamics,edu.washington.riekelab.vyom.protocols.monitor...,1,20240523C/data012,CC cone var mean noise,93,40,1858,2827,296
8,20250514C,data012,0.5,var_mean,edu.washington.riekelab.vyom.protocols.monitor...,1,20250514C/data012,CC cone var mean noise,125,40,2262,3350,491
9,20250924C,data010,0.5,var_mean,edu.washington.riekelab.chris.protocols.monito...,1,20250924C/data010,1d var mean noise,144,75,2451,3597,626


## 2. Synthetic `VariableMeanSpatialNoise` overlay

This date didn't run `VariableMeanSpatialNoise`, but we can synthesize a stim_block matching the rig (`canvasSize=[800, 600]`, `micronsPerPixel=3.8`) and feed it through the same dispatcher to see what the noise stimulus would look like overlaid on the mosaic + electrodes. Switching protocols requires no extra plumbing — `regen_stimulus` and `render_displayed_canvas` route on `stim_block.protocol_name` / `stim_ds.attrs['protocol_name']`.

In [ ]:
from types import SimpleNamespace

# Use the real rig's display params so the rendered noise lives in the same
# canvas-pixel coords as the mosaic + electrodes.
canvas_size = list(analysis_chunk.canvas_size)         # [800, 600]
mu_per_pix  = float(analysis_chunk.microns_per_pixel)  # 3.8
grid_um     = 30                                       # protocol default
stixel_um   = 90                                       # protocol default → stepsPerStixel=3
steps_per_stixel = max(round(stixel_um / grid_um), 1)
grid_size_pix   = round(grid_um / mu_per_pix)          # 8
stixel_size_pix = grid_size_pix * steps_per_stixel     # 24
n_x = int(np.ceil(canvas_size[0] / stixel_size_pix) + 1)
n_y = int(np.ceil(canvas_size[1] / stixel_size_pix) + 1)

mean_intensities = [0.03, 0.3]                          # low / high mean
total_frames = 120                                      # 2 s @ 60 Hz
frames_per_switch = 60                                  # 1 s per mean (block-level=1000 ms)

epoch_params = [{
    'seed': 42,
    'numXStixels': n_x, 'numYStixels': n_y,
    'numXChecks': n_x * steps_per_stixel, 'numYChecks': n_y * steps_per_stixel,
    'stixelSize': stixel_um, 'stepsPerStixel': steps_per_stixel,
    'frameDwell': 1, 'meanSwitchInterval': 1000,
    'meanIntensities': mean_intensities,
    'totalFrames': total_frames, 'framesPerSwitch': frames_per_switch,
    'canvasSize': canvas_size, 'micronsPerPixel': mu_per_pix, 'gridSize': grid_um,
}]
synth_stim = SimpleNamespace(
    protocol_name='edu.washington.riekelab.chris.protocols.VariableMeanSpatialNoise',
    exp_name=analysis_chunk.exp_name, datafile_name='(synthetic)',
    df_epochs=pd.DataFrame({'epoch_parameters': epoch_params}),
    d_epoch_block_params={
        'preTime': 250, 'stimTime': 2000, 'tailTime': 250,
        'contrast': 1.0, 'gridSize': grid_um, 'meanSwitchInterval': 1000,
        'meanIntensities': mean_intensities,
    },
)

# Regen → render → overlay. Same three calls as for the eye-movement protocol.
ds_vm = ra.regen_stimulus(synth_stim, verbose=True)
canvas_low  = ra.render_displayed_canvas(ds_vm, epoch=0, frame=0)    # mean=0.03
canvas_high = ra.render_displayed_canvas(ds_vm, epoch=0, frame=60)   # mean=0.30

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ra.plot_stim_with_mosaic(
    canvas_low, analysis_chunk, cell_types=cell_types, minimum_n=3,
    ax=axes[0], show_electrodes=True,
    electrode_kwargs=dict(s=5, c='cyan', edgecolors='black', linewidths=0.3),
    title=f'VM noise — mean={mean_intensities[0]} (low background)',
)
ra.plot_stim_with_mosaic(
    canvas_high, analysis_chunk, cell_types=cell_types, minimum_n=3,
    ax=axes[1], show_electrodes=True,
    electrode_kwargs=dict(s=5, c='red', edgecolors='black', linewidths=0.3),
    title=f'VM noise — mean={mean_intensities[1]} (high background)',
)
plt.tight_layout()
plt.show()